# 00 — Welcome and Setup

This series walks the EHI Atlas data platform **end-to-end, cell by cell**, against the live corpus. Each notebook covers one layer of the five-layer pipeline. Work through them in order or jump to a specific artifact deep-dive (notebooks 06–08).

**Reading order:** 00 → 01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09


## Kernel setup

Select **Python (ehi-atlas)** in VS Code / Cursor. To verify the kernel is correct, run the cell below.


In [ ]:
import ehi_atlas
print("ehi_atlas version:", ehi_atlas.__version__)


## Where the corpus lives

The corpus is a file-based three-tier store:

```
ehi-atlas/corpus/
├── _sources/    ← raw acquisitions (gitignored for personal data)
├── bronze/      ← Layer 1 output: per-source, immutable
├── silver/      ← Layer 2 output: per-source FHIR R4 bundles
├── gold/        ← Layer 3 output: merged canonical record
└── reference/   ← terminology (LOINC subset, hand-curated crosswalk)
```

From the shell:
```bash
ls ehi-atlas/corpus/bronze/
ls ehi-atlas/corpus/gold/patients/rhett759/
```


In [ ]:
from pathlib import Path

ATLAS_ROOT = Path("..").resolve()   # notebooks/ is one level below ehi-atlas/
CORPUS     = ATLAS_ROOT / "corpus"

for tier in ("bronze", "silver", "gold"):
    path = CORPUS / tier
    if path.exists():
        children = sorted(path.iterdir())
        print(f"{tier}/  ({len(children)} entries):", [c.name for c in children[:6]])
    else:
        print(f"{tier}/  — not found")


## Five-layer pipeline

```
            ┌─────────────────────────────────────────┐
            │  Source A: Synthea FHIR R4              │
            │  Source B: Epic EHI Export (SQLite)     │
            │  Source C: Synthea-Payer (Claim/EoB)    │
            │  Source D: Synthesized lab PDF          │
            │  Source E: Clinical note (FHIR)         │
            └────────────────────┬────────────────────┘
                                 │
                    ┌────────────▼─────────────┐
                    │ LAYER 1: INGEST          │  🔧 Script
                    │ Per-source adapters →    │
                    │ raw immutable bronze     │
                    └────────────┬─────────────┘
                                 │
                    ┌────────────▼─────────────┐
                    │ LAYER 2: STANDARDIZE     │  🔧 Script + 📚 Reference
                    │ Convert all sources to   │
                    │ FHIR R4 (silver)         │
                    └────────────┬─────────────┘
                                 │
                    ┌────────────▼─────────────┐
                    │ LAYER 3: HARMONIZE       │  🔧 Script + 📚 Reference
                    │ Cross-source dedup,      │  ← our differentiation
                    │ entity resolution, code  │
                    │ mapping, temporal align, │
                    │ conflict detection       │
                    └────────────┬─────────────┘
                                 │
                    ┌────────────▼─────────────┐
                    │ LAYER 4: CURATE          │  🔧 Script (existing SOF DB)
                    │ SQL-on-FHIR views        │
                    └────────────┬─────────────┘
                                 │
                    ┌────────────▼─────────────┐
                    │ LAYER 5: INTERPRET       │  ⚙️ Hybrid (existing app)
                    │ Context pipeline,        │
                    │ surgical risk, briefing  │
                    └──────────────────────────┘
```

Engine legend: 🔧 Script · 🤖 LLM · 📚 Reference · ⚙️ Hybrid


In [ ]:
# Quick sanity: gold tier for showcase patient
import json

manifest_path = CORPUS / "gold" / "patients" / "rhett759" / "manifest.json"
manifest = json.loads(manifest_path.read_text())
print("Patient:", manifest["patient_id"])
print("Sources:", [s["name"] for s in manifest["sources"]])
print("Merge summary:", manifest["merge_summary"])


**Next:** [01_bronze_tier.ipynb](./01_bronze_tier.ipynb)